Extract recipe information from cooking book and answer queries.
1. Convert pdf to image.
2. Analyze the content of the image and extract any related recipe information into structure           components.
Specifically, extra the recipe title, list of ingredients, step by step instructions, cuisine type, dish type, any relevant tags or metadata.
The output must be formatted in a way suited for embedding in a Retrieval Augmented Generation (RAG) system.

# Setup

In [57]:
# # Import the userdata module from Google Colab
# from google.colab import userdata
# # Retrieve the API key stored under 'genai_course' from Colab's userdata
# api_key = userdata.get('genai_course')
genai_key = 'your open ai api key'
api_key = genai_key

# Perform OCR and transform to images

In [26]:
# # Install the pdf2image library for converting PDF files to images
!pip install pdf2image
# # Install the poppler-utils package, required by pdf2image to work with PDF files
!apt-get install -y poppler-utils

In [27]:
# Import the libraries
from pdf2image import convert_from_path # For converting PDF files to images
import os

In [28]:
# Function to converts pdfs into images and stores the image paths
def pdf_to_images(pdf_path, output_folder):
  # Create the output folder if it doesn't exist
  if not os.path.exists(output_folder):
    os.makedirs(output_folder)

  # Convert PDF into images
  images = convert_from_path(pdf_path) # Convert each page of the PDF to an image
  image_paths = []

  # Save images and store their paths
  for i, image in enumerate(images):
    image_path = os.path.join(output_folder, f"page{i+1}.jpg") # Generate the image file path
    image.save(image_path, "JPEG") # Save the image as a JPEG file
    image_paths.append(image_path) # Append the image path to the list

  return image_paths # Return the list of image paths

In [22]:
# Define the path to the PDF and the output folder for images
pdf_path = "Things mother used to make.pdf"
output_folder = "images"

# Convert the PDF into images and store the image paths
image_paths = pdf_to_images(pdf_path, output_folder)

In [29]:
# # Install the openAI library
!pip install openai

In [30]:
# Import the libraries
from openai import OpenAI
import base64

In [31]:
# Set up connection to OpenAI API
client = OpenAI(
    api_key=api_key, # Use the provided API key for authentication
)
# Specify the model to be used
model = "gpt-4o-mini"
client = OpenAI(api_key=api_key),model='gpt-4o-mini',

In [32]:
# Read and encode one image
image_path = "images/page23.jpg" # Path to the image to be encoded

# Encode the image in base64 and decode to string
with open(image_path, "rb") as image_file:
  image_data = base64.b64encode(image_file.read()).decode('utf-8') # Encode the image in base64 and decode to string
image_data

'...'

In [33]:
# Define the system prompt
system_prompt = """
Please analyze the content of this image and extract any related recipe information.
"""

In [34]:
# Call the OpenAI API use the chat completion method
response = client.chat.completions.create(
    model = model,
    messages = [
        # Provide the system prompt
        {"role": "system", "content": system_prompt},

        # The user message contains both the text and image URL / path
        {"role": "user", "content": [
            "This is the imsage from the recipe page.",
            {"type": "image_url",
             "image_url": {"url": f"data:image/jpeg;base64,{image_data}",
                           "detail": "low"}} # Provide the image data in base64 format
        ]}
    ]
)

In [35]:
# Retrieve the content
gpt_response = response.choices[0].message.content

In [36]:
from IPython.display import Markdown, display

# Display the GPT response as Markdown
display(Markdown(gpt_response))

Here are the recipe details extracted from the image:

### Bannocks
**Ingredients:**
- 1 Cupful of Thick Sour Milk
- ½ Cupful of Sugar
- 2 Cupfuls of Flour
- ½ Cupful of Indian Meal
- 1 Teaspoonful of Soda
- A pinch of Salt

**Instructions:**
1. Make the mixture stiff enough to drop from a spoon.
2. Drop mixture, size of a walnut, into boiling fat.
3. Serve warm with maple syrup.

---

### Boston Brown Bread
**Ingredients:**
- 1 Cupful of Rye Meal
- 1 Cupful of Sour Milk
- 1 Cupful of Graham Meal
- 1 Cupful of Molasses
- 1 Cupful of Flour
- ½ Teaspoonful of Salt
- 1 Heaping Teaspoonful of Soda
- 1 Cupful of Sweet Milk

**Instructions:**
1. Stir the meals and salt together.
2. Beat the soda into the molasses until it foams; add sour milk, mix well and pour into a tin pan which has been well greased (if you have no brown-bread steamer).

Feel free to ask if you need more information!

In [37]:
# Define a function to get the GPT response and display it in Markdown
def get_gpt_response():
  gpt_response = response.choices[0].message.content # Extract the response content from the API response
  return display(Markdown(gpt_response)) # Display the response as Markdown

# Call the function to display the GPT response
get_gpt_response()

Here are the recipe details extracted from the image:

### Bannocks
**Ingredients:**
- 1 Cupful of Thick Sour Milk
- ½ Cupful of Sugar
- 2 Cupfuls of Flour
- ½ Cupful of Indian Meal
- 1 Teaspoonful of Soda
- A pinch of Salt

**Instructions:**
1. Make the mixture stiff enough to drop from a spoon.
2. Drop mixture, size of a walnut, into boiling fat.
3. Serve warm with maple syrup.

---

### Boston Brown Bread
**Ingredients:**
- 1 Cupful of Rye Meal
- 1 Cupful of Sour Milk
- 1 Cupful of Graham Meal
- 1 Cupful of Molasses
- 1 Cupful of Flour
- ½ Teaspoonful of Salt
- 1 Heaping Teaspoonful of Soda
- 1 Cupful of Sweet Milk

**Instructions:**
1. Stir the meals and salt together.
2. Beat the soda into the molasses until it foams; add sour milk, mix well and pour into a tin pan which has been well greased (if you have no brown-bread steamer).

Feel free to ask if you need more information!

In [38]:
# Define improved system prompt
system_prompt2 = """
Please analyze the content of this image and extract any related recipe information into structure components.
Specifically, extra the recipe title, list of ingredients, step by step instructions, cuisine type, dish type, any relevant tags or metadata.
The output must be formatted in a way suited for embedding in a Retrieval Augmented Generation (RAG) system.
"""

In [39]:
# Call the api to extract the information
response = client.chat.completions.create(
    model = model,
    messages = [
        # Provide the system prompt
        {"role": "system", "content": system_prompt2},

         # The user message contains both the text and image URL / path
        {"role": "user", "content": [
            "This is the image from the recipe page",
            {"type": "image_url",
             "image_url": {"url": f"data:image/jpeg;base64,{image_data}",
                           "detail": "low"}}
        ]}
    ],
    temperature = 0, # Set the temperature to 0 for deterministic output
)

In [40]:
# Print the info from the page with the improved prompt
get_gpt_response()

Here’s the structured information extracted from the recipe image:

### Recipe Title
Breads

### Ingredients
#### Bannocks
- 1 Cupful of Thick Sour Milk
- ½ Cupful of Sugar
- 2 Cupfuls of Flour
- ½ Cupful of Indian Meal
- 1 Teaspoonful of Soda
- A pinch of Salt

#### Boston Brown Bread
- 1 Cupful of Rye Meal
- 1 Cupful of Graham Meal
- 1 Cupful of Flour
- 1 Cupful of Sour Milk
- ½ Cupful of Molasses
- 1 Teaspoonful of Salt
- 1 Heaping Teaspoonful of Soda
- 1 Cupful of Sweet Milk

### Instructions
#### Bannocks
1. Make the mixture stiff enough to drop from a spoon.
2. Drop mixture, size of a walnut, into boiling fat.
3. Serve warm, with maple syrup.

#### Boston Brown Bread
1. Stir the meals and salt together.
2. Beat the soda into the molasses until it foams; add sour milk, mix well, and pour into a tin pan which has been well greased.
3. If you have no brown-bread steamer, use a regular oven.

### Cuisine Type
Traditional American

### Dish Type
Breads

### Tags/Metadata
- Quick Bread
- Breakfast
- Comfort Food
- Homemade

This format is suitable for embedding in a Retrieval Augmented Generation (RAG) system.

In [41]:
# Extract information about all of the images/recipes
extracted_recipes = []

for image_path in image_paths:
  print(f"Processing image {image_path}")

  # Reading and decoding the image
  with open(image_path, "rb") as image_file:
    image_data = base64.b64encode(image_file.read()).decode("utf-8") # Encode the image to base64 format

  # Call the API to extract the information
  response = client.chat.completions.create(
      model = model,
      messages = [
          # Provide system prompt for guidance
          {"role": "system", "content": system_prompt2},

          # The user message contains both the text and image URL / path
          {"role": "user", "content": [
              "This is the image from the recipe page", # Context for the image
              {"type": "image_url",
              "image_url": {"url": f"data:image/jpeg;base64,{image_data}", # Provide the base64 image
                            "detail": "low"}}
          ]}
      ],
      temperature = 0, # Set the temperature to 0 for deterministic output
  )

  # Extract the content and store it
  gpt_response = response.choices[0].message.content # Get the response content
  extracted_recipes.append({"image_path": image_path, "recipe_info": gpt_response}) # Store the path and extracted info
  print(f"Extracted information for {image_path}:\n{gpt_response}\n") # Print the extracted information for review

Processing image images\page1.jpg
Extracted information for images\page1.jpg:
I can't extract specific recipe information from the image you provided. However, if you have text or details from the recipe, feel free to share, and I can help you structure that information!

Processing image images\page2.jpg
Extracted information for images\page2.jpg:
I'm unable to analyze the content of the image you provided. If you can describe the recipe or provide the text, I can help you extract the relevant information.

Processing image images\page3.jpg
Extracted information for images\page3.jpg:
It seems that the image you provided does not contain any recipe information, as it appears to be a library catalog entry or a book cover. If you have a different image or specific text related to a recipe, please share that, and I can help extract the relevant information.

Processing image images\page4.jpg
Extracted information for images\page4.jpg:
I'm unable to analyze the content of the image you pro

In [42]:
# Filter out non-recipe content based on key recipe-related terms
filtered_recipes = []

for recipe in extracted_recipes:
  # Check if the extracted content contains any key recipe-related terms
  if any(keyword in recipe["recipe_info"].lower() for keyword in ["ingredients",
                                                                  "instructions",
                                                                  "recipe title"]):
     # If it does, add it to the filtered list
    filtered_recipes.append(recipe)

  # Print a message for non-recipe content
  else:
    print(f"Skipping recipe: {recipe['image_path']}")

Skipping recipe: images\page1.jpg
Skipping recipe: images\page2.jpg
Skipping recipe: images\page3.jpg
Skipping recipe: images\page4.jpg
Skipping recipe: images\page5.jpg
Skipping recipe: images\page6.jpg
Skipping recipe: images\page7.jpg
Skipping recipe: images\page8.jpg
Skipping recipe: images\page10.jpg
Skipping recipe: images\page11.jpg
Skipping recipe: images\page12.jpg
Skipping recipe: images\page16.jpg
Skipping recipe: images\page18.jpg
Skipping recipe: images\page20.jpg
Skipping recipe: images\page21.jpg
Skipping recipe: images\page22.jpg
Skipping recipe: images\page106.jpg
Skipping recipe: images\page107.jpg
Skipping recipe: images\page108.jpg
Skipping recipe: images\page133.jpg
Skipping recipe: images\page134.jpg
Skipping recipe: images\page135.jpg
Skipping recipe: images\page136.jpg


In [43]:
# import json library
import json

In [44]:
# Define the output file path
output_file = "recipe_info.json"

# Write the filtered list to a json file
with open(output_file, "w") as json_file:
  json.dump(filtered_recipes, json_file, indent = 4)

# Embeddings

In [45]:
# import libraries
import numpy as np

In [46]:
# Load the filtered recipes
with open("recipe_info.json", "r") as json_file:
  filtered_recipes = json.load(json_file)

another options would be to organize per recipe, but it should be done in the preprocessing

In [47]:
# Generate embeddings for each recipe
recipe_texts = [recipe["recipe_info"] for recipe in filtered_recipes] # Extract the text content of each recipe

# Call the API to generate embeddings for the recipe texts
embedding_response = client.embeddings.create(
    input = recipe_texts, # Provide the list of recipe texts as input
    model = "text-embedding-3-large" # Specify the embedding model to use
)

In [60]:
recipe_texts

["I can't analyze the content of the image directly. However, if you provide the text or details from the recipe, I can help you structure it into components like the title, ingredients, instructions, cuisine type, dish type, and any relevant tags.",
 'Based on the content of the image, here is the structured information extracted:\n\n### Recipe Title\n- Various Bread Recipes\n\n### List of Ingredients\n- Not specified in the image.\n\n### Step-by-Step Instructions\n- Not specified in the image.\n\n### Cuisine Type\n- Breads\n\n### Dish Type\n- Various types of bread\n\n### Relevant Tags or Metadata\n- Bannocks\n- Boston Brown Bread\n- Brown Bread (Baked)\n- Coffee Cakes\n- Corn Meal Gems\n- Cream of Tartar Biscuits\n- Crullers\n- Delicious Drop Biscuits\n- Doughnuts\n- Fried Bread\n- Graham Toast\n- Huckleberry Cake\n- Quick Graham Bread\n- Graham Bread (Raised Over Night)\n- Graham Muffins\n- Sweet Milk Griddle Cakes\n- Jenny Lind Tea Cake\n- Real Johnny Cakes\n- New England Buns\n- 

In [48]:
# Extract the embeddings
embeddings = [data.embedding for data in embedding_response.data]
embeddings

[[-0.0180976502597332,
  -0.03419090434908867,
  -0.020160142332315445,
  -0.01507653295993805,
  0.026245949789881706,
  -0.03564336523413658,
  -0.016253026202321053,
  0.0008347105467692018,
  -0.015759188681840897,
  0.023370079696178436,
  0.029949722811579704,
  -0.03904212266206741,
  -0.00749469269067049,
  0.013769319280982018,
  0.030356410890817642,
  -0.037328217178583145,
  -0.013936351984739304,
  0.031256936490535736,
  -0.010937022976577282,
  -0.054176751524209976,
  0.016456369310617447,
  -0.0006731243920512497,
  0.04235373064875603,
  0.008707497268915176,
  -0.014974861405789852,
  -0.0026289522647857666,
  -0.0375896617770195,
  0.006859241519123316,
  0.017139026895165443,
  0.019274141639471054,
  0.0104431863874197,
  0.023079587146639824,
  0.017458567395806313,
  -0.02431417815387249,
  -0.027814606204628944,
  -0.02830844186246395,
  0.008329857140779495,
  0.04287661612033844,
  0.035149529576301575,
  0.026318572461605072,
  0.026420244947075844,
  -0.010

In [49]:
# Convert the embeddings to numpy array
embedding_matrix = np.array(embeddings)
embedding_matrix

array([[-0.01809765, -0.0341909 , -0.02016014, ..., -0.00175112,
        -0.0252728 ,  0.00679025],
       [-0.00102815, -0.02743169, -0.01600541, ..., -0.00240081,
        -0.01370149,  0.02009647],
       [ 0.00573321, -0.03391951, -0.01525702, ..., -0.0016757 ,
        -0.02089225,  0.0168111 ],
       ...,
       [-0.00735199, -0.02438006, -0.01092925, ...,  0.00460259,
        -0.01173433,  0.00457221],
       [-0.00732727, -0.02763738, -0.01102125, ...,  0.00266768,
         0.00474554, -0.00371756],
       [-0.02997919, -0.03675662, -0.01425173, ..., -0.00774075,
        -0.00853327,  0.00981087]])

In [50]:
# Verify the embedding matrix
print(f"Generated embeddings for {len(filtered_recipes)} recipes.")
print(f"Each embedding is of size {len(embeddings[0])}")

Generated embeddings for 113 recipes.
Each embedding is of size 3072


Each time we retrieve information, we may get different results

# Retrieval System

In [8]:
# Install the faiss-cpu library
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 19.3 MB/s eta 0:00:00


In [61]:
# Import the faiss library
import faiss

In [62]:
# Print the embedding matrix shape
print(f"Embedding matrix shape: {embedding_matrix.shape}")

Embedding matrix shape: (113, 3072)


In [63]:
# Initialize the FAISS index for similarity search
index = faiss.IndexFlatL2(embedding_matrix.shape[1]) # Create a FAISS index with L2 distance metric
index.add(embedding_matrix) # Add the embeddings to the index

In [64]:
# Save the FAISS index to a file
faiss.write_index(index, "filtered_recipe_index.index")

In [65]:
# Save the metadata for each recipe
metadata = [{'recipe_info': recipe['recipe_info'], # Include recipe information
             'image_path': recipe['image_path']} for recipe in filtered_recipes] # Include image path

# Write metadata to a JSON file with indentation
with open("recipe_metadata.json", "w") as json_file:
  json.dump(metadata, json_file, indent = 4)

In [66]:
# Generate the embeddings for the query
query = "How to make bread?"
k = 5 # Number of top results to retrieve
query_embedding = client.embeddings.create(
    input = [query],
    model = "text-embedding-3-large"
).data[0].embedding
print(f"The query embedding is {query_embedding}\n")
query_vector = np.array(query_embedding).reshape(1, -1)  # Convert embedding to a 2D numpy array for FAISS
print(f"The query vector is {query_vector}\n")

# Search the FAISS index for the nearest neighbors
distances, indices = index.search(query_vector, min(k, len(metadata))) # Perform the search
print(f"The distances are {distances}\n")
print(f"The indices are {indices}\n")

# Store the indices and distances
stored_indices = indices[0].tolist()
stored_distances = distances[0].tolist()
print(f"The stored indices are {stored_indices}\n")
print(f"The stored distances are {stored_distances}\n")

# Print the metadata content for the top results
print("The metadata content is")
for i, dist in zip(stored_indices, stored_distances):
  if 0 <=i < len(metadata):
    print(f"Distance: {dist}, Metadata: {metadata[i]['recipe_info']}")

# Return the results
results = [(metadata[i]['recipe_info'], dist) for i, dist in zip(stored_indices, stored_distances) if 0 <= i < len(metadata)]
results # Output the results as a list of tuples containing recipe info and distance

The query embedding is [-0.019720381125807762, -0.028134725987911224, -0.022066915407776833, 0.016603518277406693, -0.04792621359229088, -0.04887430742383003, 0.038018617779016495, 0.017954552546143532, 0.0015480617294088006, 0.004168656188994646, -0.004832322709262371, -0.013889594934880733, -0.012917797081172466, -0.0015850967029109597, 0.038018617779016495, -0.0238090418279171, 0.017018308863043785, -0.016674624755978584, -0.0072233001701533794, 0.002310981974005699, -0.031856000423431396, 0.02015887387096882, 0.01501545775681734, -0.007821785286068916, 0.006275205407291651, 0.01784789189696312, 0.003410179866477847, -0.011229002848267555, -0.03960667923092842, 0.042285047471523285, -0.0036649806424975395, 0.023678677156567574, -0.034676581621170044, -0.02115437388420105, -0.015904298052191734, 0.011306035332381725, -0.01473102904856205, 0.005285630933940411, 0.00849730335175991, 0.019412249326705933, -0.01242004707455635, 0.007833636365830898, 0.00040108870598487556, -0.00825435388

[('Here’s the structured information extracted from the recipe image:\n\n### Recipe Title:\nNut Bread and Oatmeal Bread\n\n### Ingredients:\n#### Nut Bread:\n- 2½ Cups of Flour\n- 3 Teaspoons of Baking Powder\n- ¾ Cup of Milk\n- ½ Cup of Sugar\n- 1 Cup of Nuts, chopped (optional)\n\n#### Oatmeal Bread:\n- 2¾ Cups of Rolled Oats\n- 1½ Cups of Molasses\n- 1 Yeast Cake\n- Water\n\n### Instructions:\n#### Nut Bread:\n1. Mix flour, baking powder, and sugar in a bowl.\n2. Add milk and chopped nuts (if using).\n3. Stir until well combined.\n4. Pour into a greased loaf pan.\n5. Bake in a preheated oven for about twenty-five minutes.\n\n#### Oatmeal Bread:\n1. Boil water and add rolled oats and molasses.\n2. Allow to cool, then add yeast cake and mix well.\n3. Let the mixture sit until it rises.\n4. Stir again and pour into a greased loaf pan.\n5. Bake until done and let rise again.\n\n### Cuisine Type:\nAmerican\n\n### Dish Type:\nBread\n\n### Tags/Metadata:\n- Baking\n- Quick Bread\n- Homemad

In [67]:
# Define a function to query the embeddings
def query_embeddings(query, index, metadata, k = 5):
  # Generate the embeddings for the query
  query_embedding = client.embeddings.create(
      input = [query],
      model = "text-embedding-3-large"
  ).data[0].embedding
  print(f"The query embedding is {query_embedding}\n")
  query_vector = np.array(query_embedding).reshape(1, -1)
  print(f"The query vector is {query_vector}\n")

  # Search faiss index
  distances, indices = index.search(query_vector, min(k, len(metadata)))
  # print(f"The distances are {distances}\n")
  # print(f"The indices are {indices}\n")

  # Store the indices and distances
  stored_indices = indices[0].tolist()
  stored_distances = distances[0].tolist()
  print(f"The stored indices are {stored_indices}\n")
  print(f"The stored distances are {stored_distances}\n")

  # # Print the metadata content
  # print("The metadata content is")
  # for i, dist in zip(stored_indices, stored_distances):
  #   if 0 <=i < len(metadata):
  #     print(f"Distance: {dist}, Metadata: {metadata[i]['recipe_info']}")

  # Return the results
  results = [(
      metadata[i]['recipe_info'], dist) for i, dist in zip(
          stored_indices, stored_distances) if 0 <= i < len(metadata)]
  return results


In [68]:
# Test the retrieval system
query = "chocolate query"
results = query_embeddings(query, index, metadata)
print(f"The results are {results}")

The query embedding is [-0.014626344665884972, -0.005924540106207132, -0.01682029664516449, -0.006895276717841625, -0.0328744500875473, 0.009994668886065483, 0.023471800610423088, 0.0519583486020565, -0.015261894091963768, 0.046490881592035294, 0.005841831676661968, -0.01802174560725689, -0.049799222499132156, 0.0076222410425543785, -0.025787638500332832, 0.014669875614345074, -0.013781847432255745, -0.015174832195043564, -0.020424645394086838, -0.03021036647260189, 0.02106890082359314, -0.01762126199901104, -0.005419583059847355, 0.02754628285765648, 0.023367326706647873, 0.04920720309019089, -0.011326710693538189, -0.00455767335370183, -0.038620512932538986, 0.012397568672895432, 0.01265004649758339, 0.018718238919973373, 0.027911940589547157, -0.021591270342469215, -0.02025052160024643, 0.009750896133482456, 0.01760384999215603, -0.038829460740089417, 0.015897443518042564, -0.009315588511526585, 0.02211363986134529, -0.03590419515967369, -0.021643508225679398, -0.0011677134316414595

In [69]:
# Combine the results into a single string
def combined_retrived_content(results):
  combined_content = "\n\n".join([result[0] for result in results]) # Join the recipe information with double newlines
  return combined_content

# Get the combined content from results
combined_content = combined_retrived_content(results)
print(f"The combined content is {combined_content}")

The combined content is Based on the content of the image, here is the structured information extracted:

### Recipe Information

- **Recipe Title**: Chocolate Sauce
- **Cuisine Type**: Not specified
- **Dish Type**: Sauce
- **Ingredients**: Not listed in the image
- **Instructions**: Not listed in the image
- **Relevant Tags/Metadata**: Sauces, Chocolate

### Additional Recipes Listed
1. **Cranberry Sauce**
2. **Cream Mustard**
3. **Egg Sauce for Chocolate Pudding**
4. **Pudding Sauce**
5. **Sauce for Graham Pudding**

### Other Sections
- **Soups**:
  - Bean Porridge
  - Connecticut Clam Chowder
  - Massachusetts Clam Chowder
  - New England Fish Chowder
  - Lamb Broth
  - A Good Oyster Stew
  - Potato Soup

- **Vegetables**:
  - Green Corn Fritters
  - Delicious Stuffed Baked Potatoes
  - Creamed Potatoes
  - Scalloped Potatoes
  - Baked Tomatoes
  - Fried Tomatoes

This format is suitable for embedding in a Retrieval Augmented Generation (RAG) system.

Here’s the structured informa

# Generative System

In [70]:
# Define the system prompt
system_prompt3 = f"""
You are highly experienced and expert chef specialized in providing cooking advice.
Your main task is to provide information precise and accurate on the combined content.
You answer diretly to the query using only information from the provided {combined_content}.
If you don't know the answer, just say that you don't know.
Your goal is to help the user and answer the {query}
"""

In [71]:
# Define function to retrieve a response from the API
def generate_response(query, combined_content, system_prompt):
  response = client.chat.completions.create(
      model = model,
      messages = [
          {"role": "system", "content": system_prompt3}, # Provide system prompt for guidance
          {"role": "user", "content": query}, # Provide the query as user input
          {"role": "assistant", "content": combined_content} # Provide the combined content from the results
      ],
      temperature = 0, # Set temperature to 0 for deterministic output
  )
  return response

In [73]:
# Get the results from the API
query = "How to make bread?"
combined_content = combined_retrived_content(results)
response = generate_response(query, combined_content, system_prompt3)

In [74]:
# Display the outcome
get_gpt_response()

I'm sorry, but the provided content does not include a recipe for making bread. If you need a general bread recipe, I can provide one:

### Basic Bread Recipe

#### Ingredients:
- 4 cups all-purpose flour
- 2 teaspoons salt
- 2 tablespoons sugar
- 1 packet (2 1/4 teaspoons) active dry yeast
- 1 1/2 cups warm water (about 110°F)
- 2 tablespoons olive oil (optional)

#### Instructions:
1. **Activate Yeast**: In a small bowl, combine warm water, sugar, and yeast. Let it sit for about 5-10 minutes until frothy.
2. **Mix Ingredients**: In a large bowl, combine flour and salt. Make a well in the center and add the yeast mixture and olive oil.
3. **Knead Dough**: Mix until a dough forms. Transfer to a floured surface and knead for about 8-10 minutes until smooth and elastic.
4. **First Rise**: Place the dough in a greased bowl, cover with a cloth, and let it rise in a warm place for about 1-2 hours, or until doubled in size.
5. **Shape Bread**: Punch down the dough and shape it into a loaf. Place it in a greased loaf pan.
6. **Second Rise**: Cover and let it rise again for about 30-45 minutes.
7. **Preheat Oven**: Preheat your oven to 375°F (190°C).
8. **Bake**: Bake for 25-30 minutes, or until the bread is golden brown and sounds hollow when tapped on the bottom.
9. **Cool**: Remove from the oven and let it cool on a wire rack before slicing.

Feel free to ask if you need more specific information or variations!

In [75]:
# Get the results
query = "Get me the best chocolate cake recipe"
combined_content = combined_retrived_content(results)
response = generate_response(query, combined_content, system_prompt3)

In [76]:
# Display the outcome
get_gpt_response()

I'm sorry, but I don't have a specific chocolate cake recipe available. However, I can provide you with a chocolate sauce recipe if you're interested in making a sauce to accompany a cake. Would you like that?

# Rag system

In [77]:
# Build the function for Retrieval-Augmented Generation (RAG)
def rag_system(query, index, metadata, system_prompt, k = 5):
  # Retrieval System: Retrieve relevant results based on the query
  results = query_embeddings(query, index, metadata, k)

  # Content Merge: Combine the retrieved content into a single string
  combined_content = combined_retrived_content(results)

  # Generation: Generate a response based on the query and combined content
  response = generate_response(query, combined_content, system_prompt)

  # Return the generated response
  return response

In [78]:
# Test the rag system
query1 = "How to make the best chocolate cake?"
response = rag_system(query1, index, metadata, system_prompt3)
get_gpt_response()

The query embedding is [-0.002604733919724822, -0.03968388959765434, -0.013297276571393013, 0.007617205381393433, -0.019327564164996147, 0.022370068356394768, 0.00810969714075327, -0.005398256704211235, -0.006457113660871983, 0.037473149597644806, -0.0003669747384265065, 0.007113769184798002, -0.05170068517327309, -0.007600788958370686, 0.0003343129646964371, -0.04031865671277046, 0.021166199818253517, -0.004692351911216974, -0.004487146623432636, -0.015858232975006104, 0.015004580840468407, -0.02479969523847103, -0.011732247658073902, 0.03399287164211273, 0.0030698650516569614, 0.006758080795407295, 0.022610843181610107, -0.0006980386096984148, -0.036904048174619675, 0.05677882209420204, 0.04460880532860756, 0.0018906210316345096, -0.016208449378609657, -0.014720030128955841, -0.02911173366010189, 0.016919827088713646, -0.004768961574882269, -0.001930293976329267, 0.03475897014141083, 0.014402646571397781, -0.005751208867877722, 5.8526144130155444e-05, 0.002621150342747569, 0.01079104

To make the best chocolate cake, you can follow this structured recipe based on traditional methods:

### Recipe Title:
Chocolate Cake

### Ingredients:
- 1 ¾ Cups of All-Purpose Flour
- 1 ½ Cups of Sugar
- ¾ Cup of Unsweetened Cocoa Powder
- 1 ½ Teaspoons of Baking Powder
- 1 ½ Teaspoons of Baking Soda
- 1 Teaspoon of Salt
- 2 Large Eggs
- 1 Cup of Whole Milk
- ½ Cup of Vegetable Oil
- 2 Teaspoons of Vanilla Extract
- 1 Cup of Boiling Water

### Instructions:
1. Preheat your oven to 350°F (175°C). Grease and flour two 9-inch round cake pans.
2. In a large mixing bowl, combine the flour, sugar, cocoa powder, baking powder, baking soda, and salt. Mix well.
3. Add the eggs, milk, vegetable oil, and vanilla extract to the dry ingredients. Beat on medium speed for 2 minutes.
4. Carefully stir in the boiling water (the batter will be thin).
5. Pour the batter evenly into the prepared cake pans.
6. Bake for 30-35 minutes or until a toothpick inserted in the center comes out clean.
7. Allow the cakes to cool in the pans for 10 minutes, then remove from pans to cool completely on a wire rack.

### Cuisine Type:
American

### Dish Type:
Cake

### Relevant Tags/Metadata:
- Baking
- Dessert
- Chocolate
- Celebration Cake

This recipe will yield a rich and moist chocolate cake perfect for any occasion! Enjoy!

In [79]:
# Test with a different query
query2 = "I want something vegan"
response = rag_system(query2, index, metadata, system_prompt3)
get_gpt_response()

The query embedding is [-0.034548308700323105, -0.026619188487529755, -0.017104245722293854, 0.03446335345506668, -0.01730247214436531, -0.008516724221408367, 5.26128314959351e-06, -0.018930774182081223, -0.033670444041490555, 0.023518336936831474, 0.003238904057070613, -0.012955616228282452, 0.0179679524153471, -0.007178685627877712, 0.036842089146375656, -0.008821146562695503, 0.009663615375757217, -0.01138395071029663, 0.007461868692189455, -0.018406886607408524, -0.023546654731035233, 0.00599993672221899, 0.037436775863170624, -0.0069061219692230225, 0.007702574133872986, 0.008198143914341927, -0.022074105218052864, -0.012240579351782799, 0.002099093049764633, 0.01846352219581604, 0.010208742693066597, 0.024481158703565598, -0.0050618937239050865, 0.05527729541063309, -0.013698970898985863, -0.003830048255622387, 0.038739416748285294, -0.0019114842871204019, 0.015617535449564457, 0.026930689811706543, -0.04777294769883156, 0.022003307938575745, -0.024226294830441475, 0.001365472329

I don't have any vegan recipes listed in the provided content. If you're looking for vegan options, I recommend exploring plant-based ingredients or recipes that focus on vegetables, grains, and legumes. If you have specific ingredients in mind, I can help suggest a recipe or cooking method!